# Speech Models

In this notebook we deploy and test the two speech models used by the voice agent:

- **Whisper** — Speech to Text (STT) — converts audio into text
- **Higgs-Audio** — Text to Speech (TTS) — generates speech from text

Both models run on GPU (MIG slices) and are served via vLLM on OpenShift AI.

## Prerequisites

You need to be logged into OpenShift from the terminal before running these cells.

Open a **Terminal** in JupyterLab and run:

```bash
oc login --server=https://api.ocp.cloud.rhai-tmm.dev:6443 -u <username> -p <password>
oc project ai-roadshow
```

Verify you are logged in:

In [ ]:
!oc whoami
!oc project

## Create a Hugging Face Secret

The Higgs-Audio model is downloaded from Hugging Face and requires an API token.

Create a free account at https://huggingface.co and generate a token, then create the secret:

In [ ]:
%%bash
oc apply -f - <<EOF
apiVersion: v1
kind: Secret
metadata:
  name: hf-secret
type: Opaque
stringData:
  token: <your-huggingface-token>
EOF

## Speech to Text — Whisper

Whisper is the first model in the voice agent pipeline. It converts spoken audio into text.

We deploy the `whisper-large-v3-turbo` model (quantized W4A16) from the Red Hat model registry
using the KServe `LLMInferenceService` custom resource.

### Deploy Whisper

In [ ]:
!oc apply -f ../../ai-voice-agent/deploy/models/whisper/whisper-llmisvc.yaml

### Wait for Whisper to be ready

The model needs a GPU (MIG 1g.18gb slice) and takes a few minutes to pull the image and load weights.
Run the cell below to watch the pod status — wait until it shows `2/2 Running`.

In [ ]:
!oc get pods -l app.kubernetes.io/name=whisper
!echo '---'
!oc get llmisvc whisper -o jsonpath='{.status.conditions[?(@.type=="Ready")].status}'
!echo ' <- Ready status (True = ready)'

### Test Whisper

First, let's generate a short test WAV file using Python, then send it to the Whisper API.

In [ ]:
import wave
import struct
import math

# Generate a 2-second sine wave WAV file (440 Hz tone)
sample_rate = 16000
duration = 2
frequency = 440

samples = []
for i in range(sample_rate * duration):
    t = i / sample_rate
    sample = int(32767 * 0.5 * math.sin(2 * math.pi * frequency * t))
    samples.append(struct.pack('<h', sample))

with wave.open('/tmp/test.wav', 'w') as wav:
    wav.setnchannels(1)
    wav.setsampwidth(2)
    wav.setframerate(sample_rate)
    wav.writeframes(b''.join(samples))

print('Created /tmp/test.wav (2s, 16kHz, mono)')

Now get the model URL and access token, then send the audio to Whisper:

In [ ]:
import subprocess, json, base64

# Get the Whisper model URL (https)
result = subprocess.run(
    ["oc", "get", "llmisvc", "whisper", "-o",
     "jsonpath={.status.addresses[?(@.name=='gateway-external')].url}"],
    capture_output=True, text=True
)
# jsonpath returns both http and https URLs, pick https
urls = result.stdout.strip().split(" ")
model_url = [u for u in urls if u.startswith("https")][-1]

# Get the service account token
token_b64 = subprocess.run(
    ["oc", "get", "secret", "whisper-sa-whisper-sa", "-o",
     "jsonpath={.data.token}"],
    capture_output=True, text=True
).stdout.strip()

access_token = base64.b64decode(token_b64).decode()

print(f"Model URL: {model_url}")
print(f"Token: {access_token[:20]}...")

In [ ]:
import subprocess, json

result = subprocess.run(
    ["curl", "-s", "-X", "POST",
     f"{model_url}/v1/audio/transcriptions",
     "-H", f"Authorization: Bearer {access_token}",
     "-F", "file=@/tmp/test.wav",
     "-F", "model=whisper"],
    capture_output=True, text=True
)

response = json.loads(result.stdout)
print(json.dumps(response, indent=2))

Since we sent a pure sine wave tone (not speech), the transcription will likely be empty or contain
artifacts. The important thing is the API responded successfully. In the voice agent pipeline,
real speech audio is sent to this endpoint.

## Text to Speech — Higgs-Audio

Higgs-Audio is the second speech model. It generates natural-sounding speech from text,
completing the voice agent loop.

We deploy it as a standard Kubernetes Deployment with a vLLM container.
It uses a GPU (MIG 2g.35gb slice) and downloads the model from Hugging Face.

### Deploy Higgs-Audio

In [ ]:
!oc apply -f ../../ai-voice-agent/deploy/models/higgs-audio/higgs-audio-v2-deployment.yaml

### Wait for Higgs-Audio to be ready

This model downloads weights from Hugging Face on first start, so it may take several minutes.
Wait until the pod shows `1/1 Running` and the readiness probe passes.

In [ ]:
!oc get pods -l app=higgs-audio-predictor
!echo '---'
!oc rollout status deployment/higgs-audio-predictor --timeout=10s 2>&1 || echo 'Still rolling out...'

### Test Higgs-Audio

Send a text prompt to the TTS model and receive audio back.
The model returns raw PCM audio (signed 16-bit little-endian, 24kHz, mono).

In [ ]:
import subprocess, json, os

# Higgs-Audio is a plain Service (not LLMInferenceService), access via cluster DNS
higgs_url = "http://higgs-audio-predictor:8080"

result = subprocess.run(
    ["curl", "-s", "-X", "POST",
     f"{higgs_url}/v1/audio/speech",
     "-H", "Content-Type: application/json",
     "-d", json.dumps({
         "model": "higgs-audio-v2-generation-3B-base",
         "voice": "belinda",
         "input": "What would you like on your pizza?",
         "response_format": "pcm"
     }),
     "--output", "/tmp/tts-output.pcm"],
    capture_output=True, text=True
)

pcm_size = os.path.getsize('/tmp/tts-output.pcm')
print(f'Received {pcm_size} bytes of PCM audio')

Convert the raw PCM to WAV and play it in the notebook:

In [ ]:
import wave
import IPython.display as ipd

# Convert PCM (s16le, 24kHz, mono) to WAV
with open('/tmp/tts-output.pcm', 'rb') as pcm:
    pcm_data = pcm.read()

with wave.open('/tmp/tts-output.wav', 'w') as wav:
    wav.setnchannels(1)
    wav.setsampwidth(2)  # 16-bit
    wav.setframerate(24000)
    wav.writeframes(pcm_data)

print(f'Audio duration: {len(pcm_data) / (24000 * 2):.1f} seconds')
ipd.Audio('/tmp/tts-output.wav')

## Summary

You have deployed and tested both speech models:

| Model | Type | GPU | API |
|-------|------|-----|-----|
| Whisper | Speech → Text | MIG 1g.18gb | `/v1/audio/transcriptions` |
| Higgs-Audio | Text → Speech | MIG 2g.35gb | `/v1/audio/speech` |

These models form the speech layer of the voice agent pipeline:

```
User Speech → [Whisper STT] → Text → [LLM Agent] → Text → [Higgs-Audio TTS] → Speech
```

In the next section, we will deploy the LLM agent that connects these models together.